# 03 - 推测解码 (EAGLE/MTP) 原理与实践

## 学习目标
- 理解推测解码 (Speculative Decoding) 的核心思想
- 掌握 EAGLE / EAGLE3 算法的原理
- 理解 MTP (Multi-Token Prediction) 与推测解码的关系
- 学会配置 SGLang 的推测解码参数
- 实际运行对比实验，观察推测解码的加速效果

## 1. 推测解码 (Speculative Decoding) 原理

### 传统自回归解码的瓶颈

```
标准 autoregressive 解码:

Step 1: [prompt] → token_1    (1次 forward pass)
Step 2: [prompt, t1] → token_2 (1次 forward pass)
Step 3: [prompt, t1, t2] → token_3 (1次 forward pass)
...

问题: 每次只生成 1 个 token，GPU 利用率低 (memory-bound)
```

### 推测解码的思路

```
推测解码:

Step 1: Draft Model 快速生成 K 个 draft tokens: [d1, d2, d3, d4]
Step 2: Target Model 并行验证这 K 个 token（单次 forward pass）
Step 3: 从左到右找到第一个不匹配的位置，接受之前的所有 token

例:
  Draft:  [d1, d2, d3, d4]
  Target: [t1, t2, t3, t4]  (并行计算)
  Match:  [✓,  ✓,  ✓,  ✗ ]
  Accept: [t1, t2, t3] + resample t4' → 一步获得 4 个 token!
```

### 核心优势
- Target Model 只做 1 次 forward pass 就验证了多个 token
- 验证的 compute 和生成单个 token 几乎相同 (批处理 K 个位置)
- 总 token/step 从 1 提升到 **Accept Length** (平均接受的 token 数)

## 2. EAGLE 算法

EAGLE (Extrapolation Algorithm for Greater Language-model Efficiency) 是一种高效的推测解码方法。

### EAGLE vs 传统 Speculative Decoding

| 方面 | 传统方法 | EAGLE |
|------|----------|-------|
| Draft Model | 独立小模型 | **共享 Target Model 的 embedding + 轻量层** |
| 训练需求 | 需要独立训练 draft model | 只训练 1-2 层 head |
| Draft 质量 | 取决于小模型能力 | 利用 target 的特征，质量高 |
| 推测结构 | 线性链 | **树形结构 (tree)** |

### EAGLE 的核心创新

1. **特征外推** — 用 target model 的 hidden states 来预测下一个 token 的 hidden state
2. **树形推测** — 不是线性 chain，而是 tree 结构
   - 每步可以有多个候选 (topk)
   - 验证时一次性验证整棵树
   - 相比线性链，树形结构大幅提高接受率

### EAGLE3 / Multi-Layer EAGLE

EAGLE3 是 EAGLE 的进化版：
- 使用 **多层** draft head (不只 1 层)
- 每一层预测不同位置的 token (MTP Head)
- 对应 SGLang 参数: `--enable-multi-layer-eagle`

## 3. MTP (Multi-Token Prediction) 与推测解码的关系

### MTP 训练

MTP 训练目标是让模型学习「一次预测多个 token」的能力：

```
标准 LM 训练: P(token_t | token_1, ..., token_{t-1})
MTP 训练:     P(token_t, token_{t+1}, ..., token_{t+K-1} | token_1, ..., token_{t-1})
```

### MTP Head

模型在每个位置输出 K 个预测（K 个 MTP Head），每个 head 负责预测向前第 k 个 token。

```
Target Model Output (position t):
  Head 0: P(token_t)     → 标准 next-token
  Head 1: P(token_{t+1}) → 向前 2 步
  Head 2: P(token_{t+2}) → 向前 3 步
  Head 3: P(token_{t+3}) → 向前 4 步
```

### 在推测解码中的作用

MTP Head 充当 **draft model** 的角色：
1. 模型自带的 MTP heads 生成 draft tokens
2. 标准的 next-token head (Head 0) 负责验证
3. 这就是「EAGLE + MTP」在 SGLang 中的实现方式

### qf_mtp_eval 的评测目标

**评测 MTP 模型的推测解码效果**：
- 核心指标是 **Accept Length** — 每次验证平均接受多少个 token
- AL 越高 → MTP 预测越准 → 推理加速比越大
- 还可以分析每个 MTP Head 的接受率 (`mtp_head_accept_rates`)

## 4. SGLang 推测解码参数详解

### 启动命令示例

来自 `qf_mtp_eval/deploy_scripts/ds_align/start_ds_0305_eagle.sh`:

```bash
python3 -m sglang.launch_server \
    --model-path /path/to/model \
    --dp-size 8 \
    --tp-size 8 \
    --enable-dp-attention \
    --trust-remote-code \
    --mem-fraction-static 0.75 \
    --max-running-requests 256 \
    --cuda-graph-max-bs 32 \
    --chunked-prefill-size 16384 \
    --speculative-algorithm EAGLE \
    --speculative-num-steps=3 \
    --speculative-eagle-topk=1 \
    --speculative-num-draft-tokens=4 \
    --reasoning-parser deepseek-v3 \
    --host 0.0.0.0 --port 30000
```

### 参数详解

| 参数 | 默认值 | 说明 |
|------|--------|------|
| `--speculative-algorithm` | None | 推测算法名: `EAGLE`, `EAGLE3` |
| `--speculative-num-steps` | 3 | 推测步数 (tree depth) |
| `--speculative-eagle-topk` | 1 | 每步保留 top-k 候选 (tree width) |
| `--speculative-num-draft-tokens` | 4 | 每次验证的最大 draft token 数 |
| `--enable-multi-layer-eagle` | false | 是否启用多层 EAGLE (EAGLE3) |
| `--reasoning-parser` | None | 推理解析器 (deepseek-v3, glm45 等) |

### 参数之间的关系

```
树的结构:
  - 深度 = speculative-num-steps
  - 宽度 = speculative-eagle-topk
  - 总候选数 ≈ topk^steps (上界)
  - 实际验证数 = min(候选数, speculative-num-draft-tokens)

例: steps=3, topk=2
  Level 0:        [root]           (target model output)
  Level 1:      [d1]  [d2]         (2 candidates)
  Level 2:   [d3][d4] [d5][d6]     (4 candidates)
  Level 3: [..][..][..][..]...     (8 candidates)
  Total tree nodes = 2+4+8 = 14, 但 speculative-num-draft-tokens=4 会截断
```

In [ ]:
# ============================================================
# 实践：启动带 EAGLE 推测解码的 SGLang 服务
# ============================================================
import subprocess
import time
import requests

# 配置区
MODEL_PATH = "/your/mtp-model/path"  # 替换为支持 EAGLE 的模型路径
HOST = "127.0.0.1"
PORT = 30000
TP_SIZE = 8    # 根据 GPU 数调整
DP_SIZE = 1    # 数据并行

# 带推测解码的启动命令
cmd_eagle = [
    "python3", "-m", "sglang.launch_server",
    "--model-path", MODEL_PATH,
    "--host", HOST,
    "--port", str(PORT),
    "--tp-size", str(TP_SIZE),
    "--dp-size", str(DP_SIZE),
    "--trust-remote-code",
    "--mem-fraction-static", "0.75",
    "--max-running-requests", "256",
    "--cuda-graph-max-bs", "32",
    "--chunked-prefill-size", "16384",
    # === EAGLE 推测解码参数 ===
    "--speculative-algorithm", "EAGLE",
    "--speculative-num-steps", "3",
    "--speculative-eagle-topk", "1",
    "--speculative-num-draft-tokens", "4",
]

print("EAGLE launch command:")
print(" \\\n    ".join(cmd_eagle))

In [ ]:
# 启动服务（请在终端执行，或取消注释以下代码）
# server_process = subprocess.Popen(cmd_eagle, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
# print(f"EAGLE server started with PID: {server_process.pid}")

# 等待服务就绪
def wait_for_server(host, port, timeout=300):
    url = f"http://{host}:{port}/health"
    start = time.time()
    while time.time() - start < timeout:
        try:
            resp = requests.get(url, timeout=3)
            if resp.status_code == 200:
                print(f"Server ready! ({time.time()-start:.1f}s)")
                return True
        except requests.ConnectionError:
            pass
        time.sleep(5)
    return False

# wait_for_server(HOST, PORT)

## 5. 核心指标：spec_verify_ct 与 Acceptance Length

### Acceptance Length 的物理意义

```
Accept Length = completion_tokens / spec_verify_ct

含义: 每执行一次 target model 的 forward pass，平均可以获得多少个 token

- AL = 1.0: 没有推测解码效果（等价于标准解码）
- AL = 2.0: 每次验证平均接受 2 个 token（2x 加速潜力）
- AL = 3.5: 每次验证平均接受 3.5 个 token（非常好的效果）
- AL ≈ num_draft_tokens: 接近理论上限，draft 几乎全部被接受
```

### 不同粒度的 Accept Length

在 `qf_mtp_eval` 中有多种计算方式：

In [ ]:
import json
import numpy as np

# 模拟一组 meta_info 数据（实际中从 get_meta_info 获取）
sample_metas = [
    {"completion_tokens": 256, "spec_verify_ct": 80},   # AL = 3.2
    {"completion_tokens": 512, "spec_verify_ct": 170},  # AL = 3.01
    {"completion_tokens": 128, "spec_verify_ct": 45},   # AL = 2.84
    {"completion_tokens": 1024, "spec_verify_ct": 290}, # AL = 3.53
    {"completion_tokens": 64, "spec_verify_ct": 25},    # AL = 2.56
]

# --- Macro Accept Length (各样本 AL 的平均) ---
sample_als = [m["completion_tokens"] / m["spec_verify_ct"] for m in sample_metas]
macro_al = np.mean(sample_als)

# --- Micro Accept Length (全局 tokens / 全局 verify_ct) ---
total_tokens = sum(m["completion_tokens"] for m in sample_metas)
total_verify = sum(m["spec_verify_ct"] for m in sample_metas)
micro_al = total_tokens / total_verify

print("=== Accept Length Analysis ===")
print(f"\nPer-sample Accept Lengths:")
for i, al in enumerate(sample_als):
    print(f"  Sample {i}: {al:.3f}")

print(f"\nMacro AL (mean of per-sample): {macro_al:.3f}")
print(f"Micro AL (sum_tokens/sum_verify): {micro_al:.3f}")
print(f"\n注意: Macro 和 Micro 的差异来自样本长度不均")
print(f"  - 长样本对 Micro 影响更大")
print(f"  - Macro 给每个样本相同权重")

## 6. 对比实验：有无 EAGLE 的效果差异

要直观感受推测解码的加速效果，最好的方法是做对比实验。

In [ ]:
import sglang as sgl
from sglang.test.test_utils import select_sglang_backend

# 假设服务已启动（带 EAGLE）
HOST = "127.0.0.1"
PORT = 30000


@sgl.function
def bench_function(s, conv_messages):
    for msg in conv_messages:
        role = msg["role"]
        content = msg["content"]
        if role == "system":
            s += sgl.system(content)
        elif role == "user":
            s += sgl.user(content)
        elif role == "assistant":
            s += sgl.assistant(content)
    s += sgl.assistant(sgl.gen("answer"))


def run_and_measure(questions, max_tokens=512, num_threads=8):
    """Run benchmark and return metrics."""
    arguments = [{"conv_messages": q["messages"]} for q in questions]
    
    tic = time.perf_counter()
    rets = bench_function.run_batch(
        arguments,
        temperature=0,
        max_new_tokens=max_tokens,
        num_threads=num_threads,
        progress_bar=True,
    )
    latency = time.perf_counter() - tic
    
    total_tokens = sum(r.get_meta_info("answer")["completion_tokens"] for r in rets)
    throughput = total_tokens / latency
    
    # Check for speculative decoding
    has_spec = "spec_verify_ct" in rets[0].get_meta_info("answer")
    accept_length = 1.0
    if has_spec:
        als = []
        for r in rets:
            meta = r.get_meta_info("answer")
            vct = meta.get("spec_verify_ct", 0)
            if vct > 0:
                als.append(meta["completion_tokens"] / vct)
        accept_length = np.mean(als) if als else 1.0
    
    return {
        "latency": latency,
        "throughput": throughput,
        "total_tokens": total_tokens,
        "accept_length": accept_length,
        "has_speculative": has_spec,
    }

In [ ]:
# 准备测试数据
import os

DATA_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "data")
QUESTION_FILE = os.path.join(DATA_DIR, "sample_questions.jsonl")

questions = []
with open(QUESTION_FILE) as f:
    for line in f:
        obj = json.loads(line)
        questions.append(obj)

print(f"Test questions: {len(questions)}")

# 连接后端并运行
class SimpleArgs:
    backend = "srt"
    host = f"http://{HOST}"
    port = PORT
    model_path = None
    tokenizer_path = None
    base_url = None

backend = select_sglang_backend(SimpleArgs())
sgl.set_default_backend(backend)

# 运行评测
metrics = run_and_measure(questions, max_tokens=256)

print("\n=== Results ===")
print(f"Throughput:     {metrics['throughput']:.1f} tokens/s")
print(f"Latency:        {metrics['latency']:.2f}s")
print(f"Accept Length:  {metrics['accept_length']:.3f}")
print(f"Speculative:    {'Yes' if metrics['has_speculative'] else 'No'}")

## 7. 推测解码的调优指南

### 参数调优建议

| 场景 | 推荐参数 | 原因 |
|------|----------|------|
| 追求吞吐量 | steps=3, topk=1, draft=4 | 保守推测，减少浪费 |
| 追求低延迟 | steps=5, topk=2, draft=8 | 激进推测，单 request 更快 |
| 显存受限 | steps=2, topk=1, draft=3 | 减少 draft 占用 |
| 高并发场景 | steps=3, topk=1, draft=4 | 平衡 batch 效率 |

### 影响 Accept Length 的因素

1. **模型质量** — MTP Head 训练得越好，AL 越高
2. **任务类型** — 确定性高的任务 (翻译、代码) AL 更高
3. **Temperature** — temperature=0 时 AL 最高，temperature 越高 AL 越低
4. **树的形状** — 更宽的树 (topk↑) 增加接受概率，但也增加验证开销

### 加速比估算

```
理论加速比 ≈ Accept Length × (1 - draft_overhead)

其中:
  - Accept Length: 每次验证接受的平均 token 数
  - draft_overhead: draft model 的计算开销比例 (通常很小, <10%)
  
实际加速比通常为 AL 的 70%~90%
```

## 本节小结

| 知识点 | 掌握内容 |
|--------|----------|
| 推测解码原理 | Draft → Verify → Accept, 单次 forward 获得多 token |
| EAGLE 算法 | 利用 target model hidden states, 树形推测结构 |
| MTP 关系 | MTP Head 充当 draft model, AL 评估 MTP 训练效果 |
| SGLang 参数 | speculative-algorithm, num-steps, eagle-topk, num-draft-tokens |
| Accept Length | completion_tokens / spec_verify_ct, 核心评测指标 |

---
**下一节**: 04_mtp_eval_architecture — qf_mtp_eval 项目架构解析